In [ ]:
from autogen.agentchat import GroupChat, GroupChatManager, ConversableAgent, UserProxyAgent
import re

In [ ]:
import os
import sys

sys.path.append(os.path.abspath(".."))

In [ ]:
from llm_config import llm_config
from schema import schema

In [ ]:
db_schema = schema

In [ ]:
json_conversation = """
{
  "conversation": [
    {
      "user": "Show threats with highest priority",
      "assistant": {
        "type": "mixed",
        "content": {
          "text": "SELECT threat_name, priority from threat ORDER BY priority DESC",
          "table": {
            "headers": ["threat_name", "priority"],
            "rows": [
              ["Threat A", 1],
              ["Threat B", 1],
              ["Threat C", 1],
              ["Threat D", 0],
              ["Threat E", 0],
              ["Threat F", 0]
            ]
          }
        }
      }
    },
    {
      "user": "Which of those threats were detected within 48h?",
      "assistant": {
        "type": "mixed",
        "content": {
          "text": "SELECT threat_name, priority from threat WHERE event_time >= NOW() - INTERVAL '48' HOUR ORDER BY priority DESC;",
          "table": {
            "headers": ["threat_name", "priority"],
            "rows": [
              ["Threat C", 1],
              ["Threat B", 1],
              ["Threat D", 0]
            ]
          }
        }
      }
    }
  ]
}

"""

In [ ]:
investigation_agent = ConversableAgent(
    name="InvestigationAgent",
    llm_config=llm_config,
    system_message=f"""You are a cybersecurity analyst trying to investigate the details of a threat.
                      For doing so, you need to generate 3 relevant investigative questions that can be fetched from SQL database.
                      You are provided with the schema of the database {db_schema}.
                      The investigative questions that you generate should be in continuation with the previous questions and resultant table information already shared with you {json_conversation}.
                      Prepare the investigative questions in a professional tone so that it can be later queried in the database by the user"""
)
investigation_agent.description = "Helps in threat investigation"


In [ ]:
summary_agent = ConversableAgent(
    name = "SummaryAgent",
    llm_config=llm_config,
    system_message= f"""You are a cybersecurity analyst who summarize conversations and presents key data, and notable observations 
    from the table in a professional tone, as a summary to the upper management. The conversation is {json_conversation}""" 
)
summary_agent.description = "Summarize the conversation"

In [ ]:

user_proxy = UserProxyAgent(
    name="user_proxy",
    #human_input_mode="ALWAYS",  # default is 'TERMINATE'
    code_execution_config = {"use_docker": False}
)


In [ ]:


# Create the GroupChat
group_chat = GroupChat(
    agents=[investigation_agent, summary_agent],
    max_round=2, #2 needs to be fixed in this scenario, as only one agent needs to selected
    speaker_selection_method="auto"
)



In [ ]:

# Create the GroupChatManager
manager = GroupChatManager(
    groupchat=group_chat,
    llm_config = llm_config
)


In [ ]:
chat_result = user_proxy.initiate_chat(
    manager,
    message= input("Enter your request (investigate or summary): ")
)

In [ ]:

group_chat.messages

In [ ]:
chat_summary_agent = ConversableAgent(
    name = "ChatSummarizationAgent",
    llm_config=llm_config,
    system_message= f"You are a cybersecurity analyst who provide low-level observations and key findings from the contents of the chat history "
)

In [ ]:

chat_summary_prompt = f"Summarize the contents of the conversation {group_chat.messages}"


# Get the summary (assuming ConversableAgent is set up to use an LLM)
response = chat_summary_agent.generate_reply(messages=[{"role": "user", "content": chat_summary_prompt}])

print("Summary:", response)
